In [3]:
# Scientific computing.
import pandas as pd
import numpy as np
from collections import Counter

# Machine learning.
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# File handling.
from pathlib import Path
import json

In [4]:
# Anchor to the notebook's location to not hardcode paths.
Notebook_Dir = Path().resolve()
Project_Root = Notebook_Dir.parents[1]
Data_Dir = Project_Root / "Clean_Data_Resources"

# Load the parquet from Text_Parsing.ipynb.
Survey_df = pd.read_parquet(Data_Dir / "Survey_df_Text_Parsed.parquet")

# Load scale guide that maps each column/semester/discipline to its scale type.
# From Organization.ipynb.
Likert_Guide_df = pd.read_csv(Data_Dir / "Survey_Results_Likert_Guide.csv")

# Load column metadata.
# From Organization.ipynb
with open(Data_Dir / "column_metadata.json", "r") as f:
    Column_Metadata = json.load(f)

In [5]:
# Paste in Likert scales, for sanity.
# There are two different likerts that are used, where the interpretations are slightly different. 
agreement_scale = {
    'Strongly agree': 5,
    'Moderately agree': 4,
    'Neither disagree nor agree': 3,
    'Moderately disagree': 2,
    'Strongly disagree': 1,
    'Unable to judge': None
}
# Agreement with statements is not the same as assessing helpfulness directly. 
# "Moderately helpful" is not the same as "neither disagree nor agree".
helpfulness_scale = {
    'Extremely helpful': 5,
    'Very helpful': 4,
    'Moderately helpful': 3,
    'Slightly helpful': 2,
    'Not at all helpful': 1,
    'Unable to judge': None
}
# Combine both mappings in a dictionary.
rating_encoding = {**agreement_scale, **helpfulness_scale}

## Step one: Primary Pairs.
T_ Columns are paired with matching R_ columns, based on theme. 
 For instance: The open response column with understanding collaboration is paired with the equivalent rating column that also measures collaborative activities. 

In [7]:
# Leader rating columns for averaging.
Leader_R_Cols = [
    "R_Leader_Helps_Understanding_encoded",
    "R_Leader_Subject_Competence_encoded",
    "R_Leader_Has_Plan_encoded",
    "R_Leader_Willing_To_Help_encoded",
]

# T_ column to its matched R_ column.
Primary_Pairs = [
    ("T_Collaboration_Understanding",      "R_Collaborative_Activities_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Helps_Understanding_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Subject_Competence_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Has_Plan_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Willing_To_Help_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Average_encoded"),
    ("T_Other_Suggestions",                "R_Overall_Session_Helpfulness_encoded"),
    ("T_Program_Overall_Suggestions",      "R_Overall_Session_Helpfulness_encoded"),
]

In [8]:
# Create an average leader score metric.and
# T_Leader_Performance_Suggestions corresponds with columns that rate leader performance.
Survey_df["R_Leader_Average_encoded"] = Survey_df[Leader_R_Cols].mean(axis=1)

In [13]:
# Build Agreement-scale row index per R_ column from Likert_Guide_df.
# Agreement Likerts are the vast majority of responses. Helpfulness Likerts aren't.

def get_agreement_index(r_col):
    # Strip _encoded suffix to match Likert_Guide_df column names.
    col_name = r_col.replace("_encoded", "")
    if col_name == "R_Leader_Average":
        # For averaged column, intersect Agreement rows across all four leader columns.
        indices = None
        for leader_col in Leader_R_Cols:
            agreement_rows = Likert_Guide_df[
                (Likert_Guide_df["Column"] == leader_col.replace("_encoded", "")) &
                (Likert_Guide_df["Scale"] == "Agreement")
            ][["Discipline", "Course_Code", "Semester", "Year"]]
            mask = Survey_df[["Discipline", "Course_Code", "Semester", "Year"]].merge(
                agreement_rows, how="inner"
            ).index
            indices = mask if indices is None else indices.intersection(mask)
        return indices
    # For single columns, filter directly.
    agreement_rows = Likert_Guide_df[
        (Likert_Guide_df["Column"] == col_name) &
        (Likert_Guide_df["Scale"] == "Agreement")
    ][["Discipline", "Course_Code", "Semester", "Year"]]
    mask = Survey_df.merge(
        agreement_rows,
        on=["Discipline", "Course_Code", "Semester", "Year"],
        how="inner"
    ).index
    return mask